# Tutorial 03 — Building Foreground Inventories in Code

Companion explainer: **03_building_inventories.md**. We build a small electric
kettle (cradle-to-gate), link every emission to a real biosphere flow, and
**verify the LCA against a hand matrix calculation** including CH4.

In [1]:
import numpy as np
import bw2data as bd
import bw2calc as bc

bd.projects.set_current("bw25-tutorials")
BIOSPHERE = next(d for d in bd.databases if "biosphere" in d.lower())
bio = bd.Database(BIOSPHERE)


def find_flow(name, categories=("air",)):
    return next(f for f in bio
                if f["name"] == name and f["categories"] == categories)


co2 = find_flow("Carbon dioxide, fossil")
ch4 = find_flow("Methane, fossil")
print("CO2:", co2["name"], "| CH4:", ch4["name"])

CO2:

Carbon dioxide, fossil

| CH4:

Methane, fossil

## The kettle system (functional unit: 1 kettle)

electricity (coal, 1 kWh): 0.95 CO2, 0.0001 CH4
steel (1 kg): 2.9 kWh elec, 1.9 CO2
polypropylene (1 kg): 2.0 kWh elec, 1.6 CO2
kettle: 1.2 kg steel + 0.4 kg PP + 0.8 kWh elec (assembly)

In [2]:
DB = "t03_kettle"
if DB in bd.databases:
    del bd.databases[DB]

data = {
    (DB, "elec"): {
        "name": "electricity production, coal", "unit": "kilowatt hour",
        "exchanges": [
            {"input": (DB, "elec"), "amount": 1.0, "type": "production"},
            {"input": co2.key, "amount": 0.95, "type": "biosphere"},
            {"input": ch4.key, "amount": 0.0001, "type": "biosphere"},
        ],
    },
    (DB, "steel"): {
        "name": "steel production", "unit": "kilogram",
        "exchanges": [
            {"input": (DB, "steel"), "amount": 1.0, "type": "production"},
            {"input": (DB, "elec"), "amount": 2.9, "type": "technosphere"},
            {"input": co2.key, "amount": 1.9, "type": "biosphere"},
        ],
    },
    (DB, "pp"): {
        "name": "polypropylene production", "unit": "kilogram",
        "exchanges": [
            {"input": (DB, "pp"), "amount": 1.0, "type": "production"},
            {"input": (DB, "elec"), "amount": 2.0, "type": "technosphere"},
            {"input": co2.key, "amount": 1.6, "type": "biosphere"},
        ],
    },
    (DB, "kettle"): {
        "name": "kettle assembly", "unit": "unit",
        "exchanges": [
            {"input": (DB, "kettle"), "amount": 1.0, "type": "production"},
            {"input": (DB, "steel"), "amount": 1.2, "type": "technosphere"},
            {"input": (DB, "pp"), "amount": 0.4, "type": "technosphere"},
            {"input": (DB, "elec"), "amount": 0.8, "type": "technosphere"},
        ],
    },
}
bd.Database(DB).write(data)
kettle = bd.get_node(database=DB, code="kettle")
print("built kettle system:", len(bd.Database(DB)), "processes")

13:16:01-0400

 [

warning  

] 

Not able to determine geocollections for all datasets. This database is not ready for regionalization.

...

] 

Vacuuming database            

built kettle system:

4

processes

## Hand calculation (order: elec, steel, pp, kettle)

In [3]:
# A: technosphere (production +1 diagonal, inputs negative)
A = np.array([
    [1.0, -2.9, -2.0, -0.8],   # electricity
    [0.0,  1.0,  0.0, -1.2],   # steel
    [0.0,  0.0,  1.0, -0.4],   # pp
    [0.0,  0.0,  0.0,  1.0],   # kettle
])
# B: rows = [CO2, CH4]
B = np.array([
    [0.95, 1.9, 1.6, 0.0],
    [0.0001, 0.0, 0.0, 0.0],
])
f = np.array([0.0, 0.0, 0.0, 1.0])
s = np.linalg.solve(A, f)
g = B @ s
print("scaling s =", np.round(s, 4))
print("inventory: CO2 =", round(g[0], 5), "kg | CH4 =", round(g[1], 7), "kg")

# GWP100 factors (AR5): CO2=1, CH4-fossil ~= 29.8
cf = {"co2": 1.0, "ch4": 29.8}
hand_score = g[0] * cf["co2"] + g[1] * cf["ch4"]
print("hand GWP (using CF CH4=29.8) =", round(hand_score, 5), "kg CO2-eq")

scaling s =

[5.08 1.2  0.4  1.  ]

inventory: CO2 =

7.746

kg | CH4 =

0.000508

kg

hand GWP (using CF CH4=29.8) =

7.76114

kg CO2-eq

## Brightway calculation + reconciliation
The shipped method's CH4 factor may differ slightly from our 29.8; we print
both and confirm the CO2 term matches exactly.

In [4]:
gwp = next(m for m in bd.methods
           if "IPCC 2013" in str(m) and "GWP100" in str(m).replace(" ", "")
           and "no LT" not in str(m) and "SLCF" not in str(m))
lca = bc.LCA({kettle: 1}, method=gwp)
lca.lci()
lca.lcia()
print("Brightway score =", round(lca.score, 5), "kg CO2-eq")

# what CF does the shipped method use for our CH4 flow?
cfs = dict(bd.Method(gwp).load())
ch4_cf = cfs.get(ch4.id) or cfs.get(ch4.key)
co2_cf = cfs.get(co2.id) or cfs.get(co2.key)
print("shipped CFs -> CO2:", co2_cf, "| CH4:", ch4_cf)

# recompute hand score with the shipped CH4 CF
hand_exact = g[0] * (co2_cf or 1.0) + g[1] * (ch4_cf or 29.8)
print("hand score with shipped CFs =", round(hand_exact, 5))
print("relative diff =", abs(lca.score - hand_exact) / lca.score)
# ~float32 tolerance: Brightway stores exchange amounts as float32.
assert abs(lca.score - hand_exact) / lca.score < 1e-5
print("✅ hand and Brightway agree (to float32 precision) using the method's own CFs")

Brightway score =

7.76109

kg CO2-eq

shipped CFs -> CO2:

1.0

...

hand score with shipped CFs =

7.76109

relative diff =

3.660171163243459e-08

✅ hand and Brightway agree (to float32 precision) using the method's own CFs

## Mapping matrix indices back to activities

In [5]:
for act_id, col in lca.dicts.activity.items():
    print(f"col {col}: {bd.get_activity(act_id)['name']}")

col 0: electricity production, coal

col 1: steel production

col 2: polypropylene production

col 3: kettle assembly

Takeaway: when a model surprises you, densify the small foreground block and
compare against your process table. Next: **04 — importing data with bw2io**.